In [ ]:
co = spark.table("ndb.`colabfit-prod`.prod.co_po_merged_innerjoin")
softmeth = co.select("dataset_id", "method", "software").distinct()
softmeth2 = softmeth.groupby("dataset_id").agg(
    sf.collect_set("method").alias("methods_in_dataset"),
    sf.collect_set("software").alias("software_in_dataset"),
)
dsids_softmeth = softmeth2.collect()
dsids_softmeth2 = [x.asDict() for x in dsids_softmeth]

In [ ]:
dss = spark.table("ndb.`colabfit-prod`.prod.dataset_arrays")
dsids = dss.select("id")
dsids = [x["id"] for x in dsids.collect()]
len(dsids)

In [ ]:
sch = pa.schema([pa.field("software", pa.list_(pa.string()), nullable=True)])

In [ ]:
# with session.transaction() as tx:
#     table = tx.bucket('colabfit-prod').schema('prod').table('dataset_arrays')
#     table.add_column(sch)

In [ ]:
with session.transaction() as tx:
    table = tx.bucket("colabfit-prod").schema("prod").table("dataset_arrays")
    reader = table.select(columns=["id"], internal_row_id=True)
    idtab = reader.read_all()

In [ ]:
dsids_softmeth_tab = pa.Table.from_pylist(dsids_softmeth2)

combined_pd = dsids_softmeth_tab.to_pandas().merge(
    idtab.to_pandas(), left_on="dataset_id", right_on="id"
)
combined_pd.rename(
    {"methods_in_dataset": "methods", "software_in_dataset": "software"},
    axis=1,
    inplace=True,
)
combined_pd.drop("dataset_id", axis=1, inplace=True)

In [ ]:
update_table = pa.Table.from_pandas(combined_pd)

In [1]:
import pyarrow as pa
import pyarrow.parquet as pq

In [3]:
table = pq.read_table(
    "dataset_name_methods_software/part-00000-cce10d75-6fb1-44a8-9e8e-755d1d56200e-c000.snappy.parquet"
)

In [6]:
table = table.to_pandas()

In [9]:
table.to_excel("dataset_name_methods_software.xlsx", index=False)

#### Old fixes

In [ ]:
fixes = {
    "revPBE-DZ(BJ)": "DFT-revPBE+D3(BJ)",
    "R2SCAN Structure Optimization": "DFT-R2SCAN",
    "PBESol Structure Optimization": "DFT-PBEsol",
    "SCAN Structure Optimization": "DFT-SCAN",
    "GGA+U Structure Optimization": "DFT-GGA+U",
    "GGA Structure Optimization": "DFT-GGA",
    "PBEsol": "DFT-PBEsol",
    "GFN2-xTB": "IP-GFN2-xTB",
    "SchNet": "IP-SchNet",
    "mbGDML": "IP-mbGDML",
    "GAP": "IP-GAP",
    "MP2": "IP-MP2",
    "DFT-RPBE-D3": "DFT-RPBE+D3",
    "DFT-BLYP-D3": "DFT-BLYP+D3",
    "DFT-PBE-GGA": "DFT-PBE",
    "ωB97M-V": "DFT-wB97M-V",
    "DFT-revPBE0-D3": "DFT-revPBE0+D3",
    "DFT-PBE-D3": "DFT-PBE+D3",
    "DFT-vdW-optB88": "DFT-optB88-vdW",
    "B3LYP/6-31G(2df,p)": "DFT-B3LYP",
    "PBE+U": "DFT-PBE+U",
    "rPBE": "DFT-rPBE",
    "PBE": "DFT-PBE",
    "DFT-PBE-D3(BJ)": "DFT-PBE+D3(BJ)",
    "revPBE-(BJ)": "DFT-revPBE+D3(BJ)",
    "SCAN": "DFT-SCAN",
    "wB97x": "DFT-wB97X",
    "DFT-wb97x": "DFT-wB97X",
    "DFT-ωB97X": "DFT-wB97X",
    "DFT-wB97x": "DFT-wB97X",
    "DFT/optB88-vdW": "DFT-optB88-vdW",
    "DFT-OptB88vdW": "DFT-optB88-vdW",
    "DFT-PBE-TS-vdW": "DFT-PBE-vdW-TS",
    "DFT-revPBE-D3": "DFT-revPBE+D3",
    "DFT-RPBE": "DFT-rPBE",
    "DFT-wB97M-D3BJ": "DFT-wB97M+D3BJ",
    "DFT-wB97MV": "DFT-wB97M-V",
    "DFT-ωB97M-D3(BJ)": "DFT-wB97M-D3(BJ)",
    "DFT-ωB97X-D3": "DFT-wB97X+D3",
    "DFT-ωB97X-V": "DFT-wB97X-V",
    "B3LYP": "DFT-B3LYP",
}

In [ ]:
to_fix = {
    "DS_k85kj1kiekip_0": ["DFT-PBE"],
    "DS_tku3ae1rtxiy_0": ["AMBER-03", "DFT-PBE"],
}
# to_fix = {
#     "DS_4pbhjtu62o2d_0": ["DFT-PBE"]  #  references in publication point to https://arxiv.org/pdf/2007.14444, confirms PBE
# }

#### Fixes2

In [ ]:
{  # gamma fixes
    "DFT-wB97M+D3BJ": "DFT-ωB97M+D3(BJ)",
    "DFT-wB97M-D3(BJ)": "DFT-ωB97M+D3(BJ)",
    "DFT-wB97M-V": "DFT-ωB97M-V",
    "DFT-wB97X": "DFT-ωB97X",
    "DFT-wB97X+D3": "DFT-ωB97X+D3",
    "DFT-wB97X-V": "DFT-ωB97X-V",
    # minus to plus fixes
    "DFT-PBE-TS": "DFT-PBE+TS",
    "DFT-PBE-vdW-TS": "DFT-PBE+vdW-TS",
    "DFT-optB88-vdW": "DFT-optB88+vdW",
}

#### update code

In [ ]:
with session.transaction() as tx:
    table = tx.bucket("colabfit-prod").schema("prod").table("dataset_arrays")
    dstab = table.select(internal_row_id=True, columns=["id", "methods"]).read_all()
meth = dstab.to_pylist()
newmeths = []
for x in meth:
    newmeths.append(
        {
            "id": x["id"],
            "methods": [i if not i in fixes else fixes[i] for i in x["methods"]],
            "$row_id": x["$row_id"],
        }
    )

In [ ]:
id_methods2 = id_methods.to_pylist()
methods = [(x["id"], x["methods"]) for x in id_methods2]

#### datasets individual corrections

In [19]:
with open("AlNiCu.extxyz", "r") as f:
    lines = []
    for line in f:
        if "functionals" in line:
            lines.append(line)

In [20]:
methods = set(
    [
        (
            (l.split("program_name=")[1].split()[0] if "program_name" in l else None),
            (
                l.split("nomad_electronic_structure_method=")[1].split()[0]
                if "nomad_electronic_structure_method" in l
                else None
            ),
            (l.split("functionals=")[1].split()[0] if "functionals" in l else None),
        )
        for l in lines
    ]
)
print(methods)

set()


In [ ]:
functionals = {
    ("Gaussian", None, "LDA_X,LDA_C_VWN"),
    ("Gaussian", None, "HYB_MGGA_XC_M06_2X"),
    ("Gaussian", None, "B2PLYP"),
    ("Gaussian", None, "HYB_GGA_XC_TPSSh"),
    ("Gaussian", None, "HYB_GGA_XC_PBE1PBE"),
    ("Octopus", "DFT", "LDA_X,LDA_C_PZ_MOD"),
    ("Gaussian", None, "GGA_C_LYP,GGA_X_B88"),
    ("VASP", "DFT", "GGA_X_PBE,GGA_C_PBE"),
    ("Gaussian", None, "HYB_GGA_XC_B1B95"),
    ("Gaussian", None, "HYB_GGA_XC_MPW1PW91"),
    ("Gaussian", None, "GGA_C_PBE,GGA_X_PBE"),
    ("Octopus", "DFT", "GGA_X_PBE,GGA_C_PBE"),
    ("Gaussian", None, "HYB_GGA_XC_HSE06"),
    ("Gaussian", None, "HYB_GGA_XC_B3PW91"),
    ("Gaussian", None, "HYB_GGA_XC_B3LYP"),
    ("exciting", "DFT", "LDA_C_PW,LDA_X_PZ"),
}

In [ ]:
single_edits = {
    "DS_gpsibs9f47k4_0": [
        "DFT-PBE"
    ],  # probably: this is the Jarvis MP-20 dataset, which was gathered 4 years ago (2021)
    "DS_sn623uhg2d1b_0": [
        "DFT-undefined"
    ],  #  CoNbV_CMS2019 "DFT as implemented in VASP 5.4.1"
    "DS_oqqwzogut1on_0": [
        "IP-GAP"
    ],  # ndsc tut All ab initio training data and the scripts used to gen-erate the configurations are made available in a dedi-cated repository44. We used the QUIP software pack-age with the GAP plugin45, available under the GeneralPublic License and the Academic Source License, respec-tively.
    "DS_0ry6z1j8mi8c_0": ["DFT-undefined"],  # same source as DS_sn623uhg2d1b_0
    "DS_3td9plyix4ib_0": [
        "DFT-PBE",
        "DFT-HSE06",
        "DFT-mPW1PW91",
        "DFT-B1B95",
        "DFT-M06-2X",
        "DFT-B3PW91",
        "DFT-B88-LYP",
        "DFT-LDA-PW-PZ",
        "DFT-LDA-PZ_MOD",
        "DFT-LDA-C_VWN",
        "DFT-B2PLYP",
        "DFT-TPSSh",
        "DFT-PBE0",  # https://chemistry.stackexchange.com/questions/79088/pbe-vs-pbepbe-functional
    ],  # CHON JCP 2020
    "DS_vmnudsz7kx0a_0": [
        "DFT-undefined"
    ],  # AlNiCu, same source as DS_3td9plyix4ib_0. None of the stated metadata actually in data files
    "DS_xzaglubh0trq_0": ["IP-cgSchNet"],  # cgschnet - generated structures
    "DS_1s8172fnm2ct_0": [
        "DFT-undefined"
    ],  # Jarvis Materials Project. Based on/taken from materials project. Which calculations are not specified here
    "DS_gpsibs9f47k4_0": ["DFT-undefined"],  # Jarvis MP-84K. Same deal
    "DS_5drebe4tktiu_0": [
        "DFT-undefined"
    ],  # matbench dataset formation energy. From materials project data. likely pbe but not defined in data files
    "DS_dtjyh96dypuu_0": [
        "DFT-undefined"
    ],  # AlNiTi_CMS_2019 same as DS_sn623uhg2d1b_0 above
    "DS_l9f0rjjqfd67_0": ["DFT-PBE"],  # COHInPt Schaaf
    "DS_yk3t004l8dpd_0": ["DFT-PBE"],  # TdS PdV Atari500
    "DS_caktb6z8yiy7_0": ["DFT-PBE"],  # Mg edmonds 2022
    "DS_gn4qyaj4yn1x_0": [
        "CCSD(T)-F12b"
    ],  # https://static-content.springer.com/esm/art%3A10.1038%2Fnchem.2488/MediaObjects/41557_2016_BFnchem2488_MOESM407_ESM.pdf
    "DS_0sa5e1klrpzx_0": [
        "CCSD(T)",
        "MRCI",
    ],  # https://pubs.acs.org/doi/10.1021/acs.jctc.8b00298, 3.1
    "DS_82ubxqu96yz7_0": [
        "CCSD(T)",
        "MRCI",
    ],  # https://pubs.acs.org/doi/10.1021/jz200719x 5th paragraph
}

to_fix = {
    "DS_4pbhjtu62o2d_0": [
        "DFT-PBE"
    ]  #  references in publication point to https://arxiv.org/pdf/2007.14444, confirms PBE
}

In [ ]:
gammas = {'DFT-wB97M+D3BJ': "",
 'DFT-wB97M-D3(BJ)',
 'DFT-wB97M-V',
 'DFT-wB97X',
 'DFT-wB97X+D3',
 'DFT-wB97X-V'}

In [ ]:
empties = {
    "DS_720rbshv96l1_0": ["FF-ALIGN"],
    "DS_3tspv1150ejj_0": ["DFT-DSD-BLYP+D3(BJ)"],
    "DS_ctjgc03xdauc_0": ["DFT-revPBE+D3(BJ)"],
    "DS_y5m3hunroa7x_0": ["DFT-PBE"],
}

In [ ]:
for k, v in single_edits.items():
    st = str(v).replace("[", "").replace("]", "").replace("'", "").replace(" ", "")
    print(f"{k},{st}")

DS_gpsibs9f47k4_0,DFT-undefined
DS_sn623uhg2d1b_0,DFT-undefined
DS_oqqwzogut1on_0,IP-GAP
DS_0ry6z1j8mi8c_0,DFT-undefined
DS_3td9plyix4ib_0,DFT-PBE,DFT-HSE06,DFT-mPW1PW91,DFT-B1B95,DFT-M06-2X,DFT-B3PW91,DFT-B88-LYP,DFT-LDA-PW-PZ,DFT-LDA-PZ_MOD,DFT-LDA-C_VWN,DFT-B2PLYP,DFT-TPSSh,DFT-PBE0
DS_vmnudsz7kx0a_0,DFT-undefined
DS_xzaglubh0trq_0,IP-cgSchNet
DS_1s8172fnm2ct_0,DFT-undefined
DS_5drebe4tktiu_0,DFT-undefined
DS_dtjyh96dypuu_0,DFT-undefined
DS_l9f0rjjqfd67_0,DFT-PBE,MLFF
DS_yk3t004l8dpd_0,DFT-PBE
DS_caktb6z8yiy7_0,DFT-PBE


### Software

In [ ]:
{"DS_lvixye0ynk1o_0": ["Q-Chem"]}